<a href="https://colab.research.google.com/github/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/blob/main/Python%20NoteBooks/Caso_regresi%C3%B3n_polinomial_m%C3%BAltiple_con_datos_de_rendimiento_acad%C3%A9mico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contexto de los datos

Este caso de estudio utiliza los datos que se cargaron en el caso de estudio anteriormente construido en R; los datos se relacionan con el rendimiento académico que tienen estudiantes relacionados con varias variables.

Es un conjunto de datos de aproximadamente 6000 registros con la siguiente estructura de datos:

•	*horas_estudio*: Número de horas que el estudiante dedica al estudio diario entre 1 a 10.
•	*tiempo_lectura*: Tiempo dedicado específicamente a la lectura académica entre 1 a 10.
•	*horas_sueno*: Cantidad de horas de sueño diario del estudiante entre 1-9.
•	*uso_redes*: Tiempo dedicado al uso de redes sociales, de 1-9.
•	*motivacion*: Nivel de motivación académica del estudiante 1-10.
•	*asistencia*: Porcentaje de asistencia a clase 60 a 100.
•	*nivel_socioeconomico*: Nivel socioeconómico del estudiante 1-5.
•	*apoyo_familiar*: Nivel de apoyo familiar percibido 1-10.
•	*estres*: Nivel de estrés académico 1-10.

• La variable dependiente es *rendimiento_academico* como un indicador que representa el desempeño académico global del estudiante en una escala: *70.00* a *100.00*.

Los datos se pueden encontrar en el servicio github.com siguiente: raw.githubusercontent.com/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/refs/heads/main/datos/datos educacion para regresiones multiple polinomiales rendimiento_academico.csv .

El caso de estudio se puede recrean en el servicio *google collab*: https://colab.research.google.com/drive/1u_4oVVxfDOEv3A6avJs7rODqk2H49xkA?usp=sharing

De igual forma el documento *notebook* puede encontrarse en el servicio *github.com* en el espacio del autor:


# Objetivo

Crear, evaluar y comparar modelos de regresión múltiple con modelos polinomiales múltiples de segundo tercer orden y modelos *Lasso* y *Ridge*.

Se construye un modelo de regtresión múltiple y se valida el modelo para luego evaluarlo.

Posteriormente se construyen modelos polinomiales de segundo y tercer orden para ser comparado con el modelo de regresión múltiple original.

A la falta de validez en el postulado de multicolinealidad se cosntruyen modelos de regresión múltiple con datos normalizados y modelos regularizados *Lasso* y *Ridge* repectivamente.

Los modelos se aceptan si el valor de *r square ajustado* cumplen con el 85% o superior.





# Descripción

El caso de estudio sigue la misma metodología que ha sido incorporada en todos los casos anteriormente descritos y mencionada en el capítulo tres.


## Cargar librerías

Se cargan las librerías necesarias para la adecuada ejecución del caso de estudio.

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import shapiro
from scipy.stats import kstest
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import linear_reset
import statsmodels.api as sm

# Para validar posutalados
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)
from statsmodels.stats.diagnostic import (
    het_breuschpagan,
    linear_reset
)
from statsmodels.stats.stattools import (
    durbin_watson
)
from scipy.stats import (
    shapiro
)

from scipy.stats import shapiro
from scipy.stats import kstest
from scipy.stats import anderson

from sklearn.linear_model import (
    LassoCV
)

from sklearn.linear_model import (
    RidgeCV
)

from sklearn.preprocessing import (
    PolynomialFeatures
)

## Cargar funciones


In [73]:
# Funciones para implementar y evaluar modelos de regresión múltiple en Python
# Se crean regresión regresión lineal mpultiple con datos originales
# Se crean modelos de regresi[]on polinomial múltiple con datos originales
# Se crean regresión lineal múltiple, Lasso y Ridge con datos normalizados
# Rubén Pizarro Gurrola
# Mayo 2026

#========================================================
# CARGAR DATOS
#========================================================

def f_cargar_datos(ruta_archivo):

    datos = pd.read_csv(ruta_archivo)

    return datos

#========================================================
# VISUALIZAR HEAD Y TAIL
#========================================================

def f_visualizar_head_tail_reducido(
        datos,
        n = 6
):

    #----------------------------------------------------
    # Total columnas
    #----------------------------------------------------

    total_columnas = datos.shape[1]

    #----------------------------------------------------
    # Primeras 4 columnas
    #----------------------------------------------------

    idx_prim = list(
        range(
            min(4, total_columnas)
        )
    )

    #----------------------------------------------------
    # Últimas 4 columnas
    #----------------------------------------------------

    idx_ult = list(
        range(
            max(total_columnas - 4, 0),
            total_columnas
        )
    )

    #----------------------------------------------------
    # Evitar duplicados
    #----------------------------------------------------

    idx_ult = [
        i for i in idx_ult
        if i not in idx_prim
    ]

    #----------------------------------------------------
    # Subconjuntos
    #----------------------------------------------------

    datos_prim = datos.iloc[:, idx_prim]

    datos_ult = datos.iloc[:, idx_ult]

    #----------------------------------------------------
    # HEAD
    #----------------------------------------------------

    head_prim = (
        datos_prim
        .head(n)
        .astype(str)
        .reset_index(drop = True)
    )

    head_ult = (
        datos_ult
        .head(n)
        .astype(str)
        .reset_index(drop = True)
    )

    #----------------------------------------------------
    # TAIL
    #----------------------------------------------------

    tail_prim = (
        datos_prim
        .tail(n)
        .astype(str)
        .reset_index(drop = True)
    )

    tail_ult = (
        datos_ult
        .tail(n)
        .astype(str)
        .reset_index(drop = True)
    )

    #----------------------------------------------------
    # Separadores
    #----------------------------------------------------

    sep_head = pd.DataFrame({
        "...": ["..."] * n
    })

    sep_tail = pd.DataFrame({
        "...": ["..."] * n
    })

    #----------------------------------------------------
    # Combinar HEAD
    #----------------------------------------------------

    head_comb = pd.concat(

        [
            head_prim,
            sep_head,
            head_ult
        ],

        axis = 1
    )

    #----------------------------------------------------
    # Combinar TAIL
    #----------------------------------------------------

    tail_comb = pd.concat(

        [
            tail_prim,
            sep_tail,
            tail_ult
        ],

        axis = 1
    )

    #----------------------------------------------------
    # Fila separadora
    #----------------------------------------------------

    fila_sep = pd.DataFrame(

        [["..."] * head_comb.shape[1]],

        columns = head_comb.columns
    )

    #----------------------------------------------------
    # Tabla final
    #----------------------------------------------------

    tabla = pd.concat(

        [
            head_comb,
            fila_sep,
            tail_comb
        ],

        ignore_index = True
    )

    return tabla

#========================================================
# DESCRIBIR DATOS
#========================================================

def f_describir_datos(datos):

    describe = datos.describe(include = 'all')

    structure = datos.dtypes

    return {
        "describe": describe,
        "structure": structure
    }

def f_convertir_dummis(datos, variable_dependiente):

    #----------------------------------------------------------
    # f_convertir_dummis()
    #
    # Objetivo:
    #   Convertir variables categóricas y booleanas
    #   a variables dummy, manteniendo la variable
    #   dependiente al final del DataFrame.
    #
    # Argumentos:
    #   datos                : DataFrame de pandas
    #   variable_dependiente : nombre de la variable objetivo
    #
    # Valor de retorno:
    #   DataFrame transformado
    #
    #----------------------------------------------------------

    #----------------------------------------------------------
    # Copiar datos
    #----------------------------------------------------------

    datos_dummis = datos.copy()


    #----------------------------------------------------------
    # Guardar variable dependiente
    #----------------------------------------------------------

    y = datos_dummis[variable_dependiente]


    #----------------------------------------------------------
    # Eliminar temporalmente variable dependiente
    #----------------------------------------------------------

    datos_dummis = datos_dummis.drop(
        columns = [variable_dependiente]
    )


    #----------------------------------------------------------
    # Detectar variables categóricas y booleanas
    #----------------------------------------------------------

    variables_convertir = datos_dummis.select_dtypes(
        include = ["object", "bool"]
    ).columns


    #----------------------------------------------------------
    # Convertir a variables dummy
    #----------------------------------------------------------

    datos_dummis = pd.get_dummies(
        datos_dummis,
        columns = variables_convertir,
        drop_first = True,
        dtype = int
    )


    #----------------------------------------------------------
    # Agregar variable dependiente al final
    #----------------------------------------------------------

    datos_dummis[variable_dependiente] = y


    #----------------------------------------------------------
    # Devolver resultado
    #----------------------------------------------------------

    return datos_dummis

#========================================================
# PARTICIONAR DATOS
#========================================================

def f_particionar_datos(datos,
                         proporcion_entrenamiento = 0.7):

    datos_entrenamiento, datos_validacion = train_test_split(
        datos,
        train_size = proporcion_entrenamiento,
        random_state = 2026
    )

    return {
        "datos_entrenamiento": datos_entrenamiento,
        "datos_validacion": datos_validacion
    }

#========================================================
# CONVERTIR FACTOR
#========================================================

def f_convertir_factor(datos):

    datos_mod = datos.copy()

    for col in datos_mod.columns:

        if datos_mod[col].dtype == 'object':

            datos_mod[col] = datos_mod[col].astype('category')

        if datos_mod[col].dtype == 'bool':

            datos_mod[col] = datos_mod[col].astype(int)

    return datos_mod

#========================================================
# REDONDEAR VARIABLES NUMÉRICAS
#========================================================

def f_redondear_numericas(datos,
                          decimales = 2):

    datos_out = datos.copy()

    columnas_num = datos_out.select_dtypes(include = np.number).columns

    datos_out[columnas_num] = datos_out[columnas_num].round(decimales)

    return datos_out

#========================================================
# MODELO REGRESIÓN LINEAL MÚLTIPLE
#========================================================

def f_construir_modelo_RLM(datos,
                           variable_dependiente,
                           ver_resumen = True):

    X = datos.drop(columns = [variable_dependiente])

    y = datos[variable_dependiente]

    X = pd.get_dummies(X,
                       drop_first = True)

    modelo = LinearRegression()

    modelo.fit(X, y)

    if ver_resumen:

        X_sm = sm.add_constant(X)

        modelo_sm = sm.OLS(y, X_sm).fit()

        print(modelo_sm.summary())

    return modelo

#========================================================
# MODELO REGRESIÓN LINEAL MÚLTIPLE
# CON STATSMODELS
#========================================================

def f_construir_modelo_RLM_statsmodels(

        datos,

        variable_dependiente,

        ver_resumen = True
):

    #----------------------------------------------------
    # Variables independientes
    #----------------------------------------------------

    X = datos.drop(

        columns = [variable_dependiente]
    )

    #----------------------------------------------------
    # Variable dependiente
    #----------------------------------------------------

    y = datos[variable_dependiente]

    #----------------------------------------------------
    # Copia directa
    # SIN VARIABLES DUMMY
    #----------------------------------------------------

    X = X.copy()

    #----------------------------------------------------
    # Constante
    #----------------------------------------------------

    X = sm.add_constant(

        X,

        has_constant = "add"
    )

    #----------------------------------------------------
    # Modelo OLS
    #----------------------------------------------------

    modelo = sm.OLS(

        y,

        X

    ).fit()

    #----------------------------------------------------
    # Resumen
    #----------------------------------------------------

    if ver_resumen:

        print(

            modelo.summary()
        )

    return modelo

#========================================================
# REGRESIÓN MÚLTIPLE POLINOMIAL
# CON STATSMODELS
#========================================================

from sklearn.preprocessing import (
    PolynomialFeatures
)

def f_multiple_polinomial(

        datos,

        variable_dependiente,

        grado = 2,

        ver_resumen = True

):

    #----------------------------------------------------
    # Variables independientes
    #----------------------------------------------------

    X = datos.drop(

        columns = [variable_dependiente]
    )

    #----------------------------------------------------
    # Variable dependiente
    #----------------------------------------------------

    y = datos[variable_dependiente]

    #----------------------------------------------------
    # Variables originales
    #----------------------------------------------------

    nombres_originales = X.columns

    #----------------------------------------------------
    # Transformación polinomial
    #----------------------------------------------------

    poly = PolynomialFeatures(

        degree = grado,

        include_bias = False
    )

    X_poly = poly.fit_transform(

        X
    )

    #----------------------------------------------------
    # Nombres variables polinomiales
    #----------------------------------------------------

    nombres_poly = poly.get_feature_names_out(

        nombres_originales
    )

    #----------------------------------------------------
    # DataFrame polinomial
    #----------------------------------------------------

    X_poly = pd.DataFrame(

        X_poly,

        columns = nombres_poly,

        index = X.index
    )

    #----------------------------------------------------
    # Constante
    #----------------------------------------------------

    X_poly = sm.add_constant(

        X_poly,

        has_constant = "add"
    )

    #----------------------------------------------------
    # Modelo OLS
    #----------------------------------------------------

    modelo = sm.OLS(

        y,

        X_poly

    ).fit()

    #----------------------------------------------------
    # Guardar estructura
    # IMPORTANTÍSIMO
    #----------------------------------------------------

    modelo.columnas_entrenamiento = X_poly.columns

    modelo.poly_transformador = poly

    modelo.nombres_originales = nombres_originales

    #----------------------------------------------------
    # Información
    #----------------------------------------------------

    print("\n============================")

    print(

        f"MODELO POLINOMIAL GRADO {grado}"
    )

    print("============================")

    print(

        f"Número de variables originales: "
        f"{len(nombres_originales)}"
    )

    print(

        f"Número de términos polinomiales: "
        f"{X_poly.shape[1]-1}"
    )

    #----------------------------------------------------
    # Resumen
    #----------------------------------------------------

    if ver_resumen:

        print(

            modelo.summary()
        )

    return modelo

#========================================================
# ESTANDARIZAR Y ESCALAR
#========================================================

def f_estandarizar_escalar(datos,
                           decimales = 4):

    datos_est = datos.copy()
    datos_esc = datos.copy()

    columnas_num = datos.select_dtypes(include = np.number).columns

    scaler_est = StandardScaler()

    scaler_minmax = MinMaxScaler()

    datos_est[columnas_num] = np.round(
        scaler_est.fit_transform(datos[columnas_num]),
        decimales
    )

    datos_esc[columnas_num] = np.round(
        scaler_minmax.fit_transform(datos[columnas_num]),
        decimales
    )

    return {
        "datos_estandarizados": datos_est,
        "datos_escalados": datos_esc
    }

#========================================================
# MODELO LASSO
#========================================================


def f_construir_modelo_lasso(

        datos,

        variable_dependiente,

        cv = 10,

        random_state = 2026,

        ver_resumen = True

):

    #----------------------------------------------------
    # Variables independientes
    #----------------------------------------------------

    X = datos.drop(

        columns = [variable_dependiente]
    )

    #----------------------------------------------------
    # Variable dependiente
    #----------------------------------------------------

    y = datos[variable_dependiente]

    #----------------------------------------------------
    # Modelo LASSO CV
    #----------------------------------------------------

    modelo = LassoCV(

        cv = cv,

        random_state = random_state

    )

    modelo.fit(

        X,

        y
    )

    #----------------------------------------------------
    # Coeficientes
    #----------------------------------------------------

    coeficientes = pd.DataFrame({

        "Variable":

            X.columns,

        "Coeficiente":

            np.round(
                modelo.coef_,
                4
            )
    })

    #----------------------------------------------------
    # Información
    #----------------------------------------------------

    if ver_resumen:

        print("\n============================")

        print("MODELO LASSO")

        print("============================")

        print(

            f"Lambda óptimo: "
            f"{round(modelo.alpha_,4)}"
        )

        print(

            f"Intercepto: "
            f"{round(modelo.intercept_,4)}"
        )

        print("\nCoeficientes:")

        print(coeficientes)

    #----------------------------------------------------
    # Guardar columnas
    #----------------------------------------------------

    modelo.columnas_entrenamiento = X.columns

    return modelo

  #========================================================
# MODELO RIDGE
#========================================================



def f_construir_modelo_ridge(
        datos,
        variable_dependiente,
        alphas = np.logspace(-4, 4, 100),
        cv = 10,
        ver_resumen = True
):

    #----------------------------------------------------
    # Variables independientes
    #----------------------------------------------------

    X = datos.drop(

        columns = [variable_dependiente]
    )

    #----------------------------------------------------
    # Variable dependiente
    #----------------------------------------------------

    y = datos[variable_dependiente]

    #----------------------------------------------------
    # Modelo RIDGE CV
    #----------------------------------------------------

    modelo = RidgeCV(

        alphas = alphas,

        cv = cv
    )

    modelo.fit(

        X,

        y
    )

    #----------------------------------------------------
    # Coeficientes
    #----------------------------------------------------

    coeficientes = pd.DataFrame({

        "Variable":

            X.columns,

        "Coeficiente":

            np.round(
                modelo.coef_,
                4
            )
    })

    #----------------------------------------------------
    # Información
    #----------------------------------------------------

    if ver_resumen:

        print("\n============================")

        print("MODELO RIDGE")

        print("============================")

        print(

            f"Lambda óptimo: "
            f"{round(modelo.alpha_,4)}"
        )

        print(

            f"Intercepto: "
            f"{round(modelo.intercept_,4)}"
        )

        print("\nCoeficientes:")

        print(coeficientes)

    #----------------------------------------------------
    # Guardar columnas entrenamiento
    #----------------------------------------------------

    modelo.columnas_entrenamiento = X.columns

    return modelo

#========================================================
# MULTICOLINEALIDAD VIF
#========================================================

def f_multicolinealidad(datos,
                        variable_dependiente):

    X = datos.drop(columns = [variable_dependiente])

    X = pd.get_dummies(X,
                       drop_first = True)

    X = sm.add_constant(X)

    vif = pd.DataFrame()

    vif["Variable"] = X.columns

    vif["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif

#========================================================
# DIAGNÓSTICO DE LINEALIDAD
#========================================================

def f_linealidad(modelo):

    #----------------------------------------------------
    # Valores ajustados
    #----------------------------------------------------

    ajustados = modelo.fittedvalues

    #----------------------------------------------------
    # Residuos
    #----------------------------------------------------

    residuos = modelo.resid

    #----------------------------------------------------
    # Gráfico
    #----------------------------------------------------

    plt.figure(

        figsize = (6,4)
    )

    plt.scatter(

        ajustados,

        residuos,

        color = "blue"
    )

    plt.axhline(

        0,

        linestyle = "--",

        color = "red"
    )

    plt.title(

        "Residuos vs Valores Ajustados"
    )

    plt.xlabel(

        "Valores ajustados"
    )

    plt.ylabel(

        "Residuos"
    )

    plt.show()

    #----------------------------------------------------
    # Interpretación
    #----------------------------------------------------

    print("\n============================")

    print("Diagnóstico de Linealidad")

    print("============================")

    print(
        "- Los residuos deben distribuirse "
        "aleatoriamente alrededor de 0."
    )

    print(
        "- No deben observarse patrones "
        "curvos."
    )

    print(
        "- Curvaturas sugieren "
        "no linealidad."
    )

#========================================================
# TEST DE LINEALIDAD
# RAMSEY RESET
#========================================================

def f_linealidad_test(

        modelo

):

    #----------------------------------------------------
    # Ramsey RESET
    #----------------------------------------------------

    resultado = linear_reset(

        modelo,

        power = 2,

        use_f = True
    )

    #----------------------------------------------------
    # Mostrar resultado
    #----------------------------------------------------

    print("\n============================")

    print("Test de Linealidad (Ramsey RESET)")

    print("============================")

    print(resultado)

    #----------------------------------------------------
    # Interpretación
    #----------------------------------------------------

    p_valor = resultado.pvalue

    print("\nInterpretación:")

    if p_valor > 0.05:

        print(
            "✔ No se rechaza H0 → "
            "El modelo es lineal "
            "(no hay evidencia de curvatura)"
        )

    else:

        print(
            "❌ Se rechaza H0 → "
            "Existe evidencia de no linealidad"
        )

    return resultado

#========================================================
# HOMOCEDASTICIDAD
#========================================================

def f_homocedasticidad(

        modelo

):

    #----------------------------------------------------
    # Residuos
    #----------------------------------------------------

    residuos = modelo.resid

    #----------------------------------------------------
    # Valores ajustados
    #----------------------------------------------------

    ajustados = modelo.fittedvalues

    #----------------------------------------------------
    # Matriz X usada en el modelo
    #----------------------------------------------------

    X = modelo.model.exog

    #----------------------------------------------------
    # Breusch-Pagan
    #----------------------------------------------------

    bp = het_breuschpagan(

        residuos,

        X
    )

    #----------------------------------------------------
    # Resultados
    #----------------------------------------------------

    resultado = pd.DataFrame({

        "Prueba": [

            "Breusch-Pagan"
        ],

        "LM Statistic": [

            round(bp[0], 4)
        ],

        "LM p-value": [

            round(bp[1], 4)
        ],

        "F Statistic": [

            round(bp[2], 4)
        ],

        "F p-value": [

            round(bp[3], 4)
        ]
    })

    #----------------------------------------------------
    # Gráfico
    #----------------------------------------------------

    plt.figure(

        figsize = (6,4)
    )

    plt.scatter(

        ajustados,

        residuos,

        color = "blue"
    )

    plt.axhline(

        0,

        linestyle = "--",

        color = "red"
    )

    plt.title(

        "Residuos vs Valores Ajustados"
    )

    plt.xlabel(

        "Valores ajustados"
    )

    plt.ylabel(

        "Residuos"
    )

    plt.show()

    #----------------------------------------------------
    # Interpretación
    #----------------------------------------------------

    print("\n============================")

    print("Diagnóstico de Homocedasticidad")

    print("============================")

    print(resultado)

    print("\nInterpretación:")

    if bp[1] > 0.05:

        print(
            "✔ No se rechaza H0 → "
            "Existe homocedasticidad"
        )

    else:

        print(
            "❌ Se rechaza H0 → "
            "Existe heterocedasticidad"
        )

    return resultado

#========================================================
# NORMALIDAD DE RESIDUOS
#========================================================

def f_normalidad(

        modelo

):

    #----------------------------------------------------
    # Residuos estandarizados
    #----------------------------------------------------

    residuos = modelo.resid

    residuos = (

        residuos - np.mean(residuos)

    ) / np.std(residuos)

    #----------------------------------------------------
    # SHAPIRO-WILK
    #----------------------------------------------------

    shapiro_test = shapiro(

        residuos
    )

    #----------------------------------------------------
    # KOLMOGOROV-SMIRNOV
    #----------------------------------------------------

    ks_test = kstest(

        residuos,

        'norm'
    )

    #----------------------------------------------------
    # ANDERSON-DARLING
    #----------------------------------------------------

    ad_test = anderson(

        residuos,

        dist = 'norm'
    )

    #----------------------------------------------------
    # Aproximación interpretación
    # scipy no devuelve p-value directo
    # usamos nivel 5%
    #----------------------------------------------------

    ad_critico_5 = ad_test.critical_values[2]

    ad_interpretacion = (

        "Normalidad"

        if ad_test.statistic < ad_critico_5

        else

        "No normalidad"
    )

    #----------------------------------------------------
    # RESULTADOS
    #----------------------------------------------------

    resultado = pd.DataFrame({

        "Prueba": [

            "Shapiro-Wilk",

            "Kolmogorov-Smirnov",

            "Anderson-Darling"
        ],

        "Estadistico": [

            round(shapiro_test.statistic, 4),

            round(ks_test.statistic, 4),

            round(ad_test.statistic, 4)
        ],

        "p_value": [

            round(shapiro_test.pvalue, 4),

            round(ks_test.pvalue, 4),

            np.nan
        ],

        "Interpretacion": [

            "Normalidad"

            if shapiro_test.pvalue > 0.05

            else

            "No normalidad",

            "Normalidad"

            if ks_test.pvalue > 0.05

            else

            "No normalidad",

            ad_interpretacion
        ]
    })

    #----------------------------------------------------
    # HISTOGRAMA
    #----------------------------------------------------

    plt.figure(

        figsize = (6,4)
    )

    plt.hist(

        residuos,

        bins = 15,

        density = True,

        alpha = 0.7
    )

    plt.title(

        "Histograma de residuos"
    )

    plt.xlabel(

        "Residuos estandarizados"
    )

    plt.ylabel(

        "Frecuencia"
    )

    plt.show()

    #----------------------------------------------------
    # QQ-PLOT
    #----------------------------------------------------

    sm.qqplot(

        residuos,

        line = '45'
    )

    plt.title(

        "QQ-Plot residuos"
    )

    plt.show()

    #----------------------------------------------------
    # RESULTADOS
    #----------------------------------------------------

    print("\n============================")

    print("Diagnóstico de Normalidad")

    print("============================")

    print(resultado)

    #----------------------------------------------------
    # Anderson detalle
    #----------------------------------------------------

    print("\nAnderson-Darling:")

    print(

        f"Estadístico = "
        f"{round(ad_test.statistic,4)}"
    )

    print(

        f"Valor crítico 5% = "
        f"{round(ad_critico_5,4)}"
    )

    print("\nInterpretación:")

    for i in range(len(resultado)):

        prueba = resultado.iloc[i]["Prueba"]

        interpretacion = resultado.iloc[i]["Interpretacion"]

        if interpretacion == "Normalidad":

            print(

                f"✔ {prueba}: Normalidad"
            )

        else:

            print(

                f"❌ {prueba}: No normalidad"
            )

    return resultado


#========================================================
# INDEPENDENCIA DE RESIDUOS
#========================================================

from statsmodels.stats.stattools import durbin_watson

def f_independencia(

        modelo

):

    #----------------------------------------------------
    # Residuos
    #----------------------------------------------------

    residuos = modelo.resid

    #----------------------------------------------------
    # Durbin-Watson
    #----------------------------------------------------

    dw = durbin_watson(

        residuos
    )

    #----------------------------------------------------
    # DataFrame resultado
    #----------------------------------------------------

    resultado = pd.DataFrame({

        "Prueba": [

            "Durbin-Watson"
        ],

        "Estadistico": [

            round(dw,4)
        ],

        "Interpretacion": [

            "Sin autocorrelación"

            if 1.5 <= dw <= 2.5

            else

            "Posible autocorrelación"
        ]
    })

    #----------------------------------------------------
    # Gráfico residuos
    #----------------------------------------------------

    plt.figure(

        figsize = (7,4)
    )

    plt.plot(

        residuos,

        color = "black",

        linewidth = 1
    )

    plt.axhline(

        0,

        linestyle = "--",

        color = "red"
    )

    plt.title(

        "Residuos en secuencia"
    )

    plt.xlabel(

        "Observación"
    )

    plt.ylabel(

        "Residuo"
    )

    plt.show()

    #----------------------------------------------------
    # Resultados
    #----------------------------------------------------

    print("\n============================")

    print("Diagnóstico de Independencia")

    print("============================")

    print(resultado)

    #----------------------------------------------------
    # Interpretación ampliada
    #----------------------------------------------------

    print("\nInterpretación:")

    if 1.5 <= dw <= 2.5:

        print(
            "✔ No existe evidencia "
            "de autocorrelación"
        )

    elif dw < 1.5:

        print(
            "❌ Existe posible "
            "autocorrelación positiva"
        )

    else:

        print(
            "❌ Existe posible "
            "autocorrelación negativa"
        )

    return resultado



#========================================================
# VALIDAR POSTULADOS MULTIMODELO
#========================================================

#========================================================
# VALIDAR POSTULADOS
# OLS + LASSO + RIDGE
#========================================================

def f_validar_postulados_modelos(

        modelos,

        X_datos = None,

        y_datos = None,

        nombres = None,

        redondeo = 4

):

    #----------------------------------------------------
    # Lista modelos
    #----------------------------------------------------

    if not isinstance(modelos, list):

        modelos = [modelos]

    #----------------------------------------------------
    # Nombres
    #----------------------------------------------------

    if nombres is None:

        nombres = [

            f"Modelo_{i+1}"

            for i in range(len(modelos))
        ]

    #----------------------------------------------------
    # Resultados
    #----------------------------------------------------

    resultados = []

    #====================================================
    # RECORRER MODELOS
    #====================================================

    for i, modelo in enumerate(modelos):

        #================================================
        # MODELOS OLS STATSMODELS
        #================================================

        if hasattr(modelo, "model"):

            X = modelo.model.exog

            residuos = modelo.resid

            #--------------------------------------------
            # VIF
            #--------------------------------------------

            vif_valores = []

            for j in range(1, X.shape[1]):

                vif = variance_inflation_factor(

                    X,

                    j
                )

                vif_valores.append(vif)

            vif_max = max(vif_valores)

            #--------------------------------------------
            # Interpretación VIF
            #--------------------------------------------

            if vif_max < 5:

                interpretacion_vif = "Cumple"

            elif vif_max < 10:

                interpretacion_vif = "Moderada"

            else:

                interpretacion_vif = "No cumple"

            #--------------------------------------------
            # Linealidad
            #--------------------------------------------

            reset = linear_reset(

                modelo,

                power = 2,

                use_f = True
            )

            p_lineal = reset.pvalue

            linealidad = (

                "Cumple"

                if p_lineal > 0.05

                else

                "No cumple"
            )

            #--------------------------------------------
            # Homocedasticidad
            #--------------------------------------------

            bp = het_breuschpagan(

                residuos,

                X
            )

            p_homo = bp[1]

            homocedasticidad = (

                "Cumple"

                if p_homo > 0.05

                else

                "No cumple"
            )

        #================================================
        # MODELOS LASSO / RIDGE
        #================================================

        else:

            #--------------------------------------------
            # Validación
            #--------------------------------------------

            if X_datos is None or y_datos is None:

                raise ValueError(

                    "LASSO/RIDGE requieren "
                    "X_datos e y_datos"
                )

            #--------------------------------------------
            # Predicciones
            #--------------------------------------------

            pred = modelo.predict(

                X_datos
            )

            residuos = y_datos - pred

            #--------------------------------------------
            # VIF
            #--------------------------------------------

            vif_valores = []

            for j in range(X_datos.shape[1]):

                vif = variance_inflation_factor(

                    X_datos.values,

                    j
                )

                vif_valores.append(vif)

            vif_max = max(vif_valores)

            #--------------------------------------------
            # Interpretación VIF
            #--------------------------------------------

            if vif_max < 5:

                interpretacion_vif = "Cumple"

            elif vif_max < 10:

                interpretacion_vif = "Moderada"

            else:

                interpretacion_vif = "No cumple"

            #--------------------------------------------
            # No aplica formalmente
            #--------------------------------------------

            linealidad = "NA"

            homocedasticidad = "NA"

        #================================================
        # NORMALIDAD
        #================================================

        shapiro_test = shapiro(

            residuos
        )

        normalidad = (

            "Cumple"

            if shapiro_test.pvalue > 0.05

            else

            "No cumple"
        )

        #================================================
        # INDEPENDENCIA
        #================================================

        dw = durbin_watson(

            residuos
        )

        independencia = (

            "Cumple"

            if 1.5 <= dw <= 2.5

            else

            "No cumple"
        )

        #================================================
        # RESULTADO
        #================================================

        resultados.append({

            "Modelo":

                nombres[i],

            "Multicolinealidad":

                interpretacion_vif,

            "VIF_Max":

                round(vif_max, redondeo),

            "Linealidad":

                linealidad,

            "Homocedasticidad":

                homocedasticidad,

            "Normalidad":

                normalidad,

            "Independencia":

                independencia
        })

    #----------------------------------------------------
    # DataFrame final
    #----------------------------------------------------

    resultados = pd.DataFrame(

        resultados
    )

    #----------------------------------------------------
    # Mostrar
    #----------------------------------------------------

    print("\n============================")

    print("Validación de Postulados")

    print("============================")

    print(resultados)

    return resultados

#========================================================
# ECUACIÓN DEL MODELO
#========================================================

#========================================================
# ECUACIONES DE MODELOS
# OLS + LASSO + RIDGE
#========================================================

def f_ecuaciones_modelos(

        modelos,

        nombres = None,

        redondeo = 4

):

    #----------------------------------------------------
    # Convertir a lista
    #----------------------------------------------------

    if not isinstance(modelos, list):

        modelos = [modelos]

    #----------------------------------------------------
    # Nombres modelos
    #----------------------------------------------------

    if nombres is None:

        nombres = [

            f"Modelo_{i+1}"

            for i in range(len(modelos))
        ]

    #----------------------------------------------------
    # Lista ecuaciones
    #----------------------------------------------------

    ecuaciones = []

    #====================================================
    # RECORRER MODELOS
    #====================================================

    for i, modelo in enumerate(modelos):

        print("\n============================")

        print(

            f"ECUACIÓN: {nombres[i]}"
        )

        print("============================\n")

        #================================================
        # MODELOS OLS (STATSMODELS)
        #================================================

        if hasattr(modelo, "params"):

            coefs = modelo.params.round(

                redondeo
            )

            intercepto = coefs.iloc[0]

            variables = coefs.index[1:]

            valores = coefs.iloc[1:]

        #================================================
        # MODELOS LASSO / RIDGE
        #================================================

        else:

            intercepto = round(

                modelo.intercept_,

                redondeo
            )

            variables = modelo.columnas_entrenamiento

            valores = np.round(

                modelo.coef_,

                redondeo
            )

        #================================================
        # ECUACIÓN
        #================================================

        ecuacion = f"ŷ = {intercepto}"

        #------------------------------------------------
        # Construcción términos
        #------------------------------------------------

        for variable, valor in zip(

                variables,

                valores
        ):

            signo = (

                " + "

                if valor >= 0

                else

                " - "
            )

            termino = (

                f"{signo}"

                f"{abs(valor)}"

                f"*{variable}"
            )

            ecuacion += termino

        #------------------------------------------------
        # Mostrar
        #------------------------------------------------

        print(ecuacion)

        #------------------------------------------------
        # Guardar
        #------------------------------------------------

        ecuaciones.append({

            "Modelo":

                nombres[i],

            "Ecuacion":

                ecuacion
        })

    #----------------------------------------------------
    # DataFrame final
    #----------------------------------------------------

    ecuaciones = pd.DataFrame(

        ecuaciones
    )

    return ecuaciones

#========================================================
# EVALUACIÓN MULTIMODELO
# OLS + LASSO + RIDGE
#========================================================

from sklearn.metrics import (

    mean_squared_error,

    mean_absolute_error,

    r2_score
)

#========================================================
# EVALUACIÓN MULTIMODELO
# OLS + POLINOMIAL + LASSO + RIDGE
#========================================================

from sklearn.metrics import (

    mean_squared_error,

    mean_absolute_error,

    r2_score
)

def f_evaluacion_modelos(

        modelos,

        datos_validacion,

        variable_dependiente,

        nombres = None,

        redondeo = 4

):

    #----------------------------------------------------
    # Convertir a lista
    #----------------------------------------------------

    if not isinstance(modelos, list):

        modelos = [modelos]

    #----------------------------------------------------
    # Nombres modelos
    #----------------------------------------------------

    if nombres is None:

        nombres = [

            f"Modelo_{i+1}"

            for i in range(len(modelos))
        ]

    #----------------------------------------------------
    # Resultados
    #----------------------------------------------------

    resultados = []

    #====================================================
    # DATOS VALIDACIÓN
    #====================================================

    y_real = datos_validacion[

        variable_dependiente
    ]

    X_val = datos_validacion.drop(

        columns = [variable_dependiente]
    )

    #====================================================
    # RECORRER MODELOS
    #====================================================

    for i, modelo in enumerate(modelos):

        #================================================
        # MODELOS OLS / POLINOMIALES
        #================================================

        if hasattr(modelo, "model"):

            #--------------------------------------------
            # MODELO POLINOMIAL
            #--------------------------------------------

            if hasattr(modelo, "poly_transformador"):

                #----------------------------------------
                # Variables originales
                #----------------------------------------

                X_base = X_val[

                    modelo.nombres_originales
                ]

                #----------------------------------------
                # Transformación polinomial
                #----------------------------------------

                X_poly = modelo.poly_transformador.transform(

                    X_base
                )

                #----------------------------------------
                # Nombres polinomiales
                #----------------------------------------

                nombres_poly = (

                    modelo.poly_transformador
                    .get_feature_names_out(

                        modelo.nombres_originales
                    )
                )

                #----------------------------------------
                # DataFrame
                #----------------------------------------

                X_pred = pd.DataFrame(

                    X_poly,

                    columns = nombres_poly,

                    index = X_base.index
                )

            #--------------------------------------------
            # MODELO LINEAL NORMAL
            #--------------------------------------------

            else:

                X_pred = X_val.copy()

            #--------------------------------------------
            # Constante
            #--------------------------------------------

            X_pred = sm.add_constant(

                X_pred,

                has_constant = "add"
            )

            #--------------------------------------------
            # Reordenar columnas
            #--------------------------------------------

            X_pred = X_pred.reindex(

                columns = modelo.model.exog_names,

                fill_value = 0
            )

            #--------------------------------------------
            # Predicciones
            #--------------------------------------------

            pred = modelo.predict(

                X_pred
            )

            p = X_pred.shape[1] - 1

        #================================================
        # MODELOS LASSO / RIDGE
        #================================================

        else:

            #--------------------------------------------
            # Copia directa
            #--------------------------------------------

            X_pred = X_val.copy()

            #--------------------------------------------
            # Reordenar columnas
            #--------------------------------------------

            X_pred = X_pred.reindex(

                columns = modelo.columnas_entrenamiento,

                fill_value = 0
            )

            #--------------------------------------------
            # Predicción
            #--------------------------------------------

            pred = modelo.predict(

                X_pred
            )

            p = X_pred.shape[1]

        #================================================
        # MÉTRICAS
        #================================================

        mse = mean_squared_error(

            y_real,

            pred
        )

        rmse = np.sqrt(mse)

        mae = mean_absolute_error(

            y_real,

            pred
        )

        r2 = r2_score(

            y_real,

            pred
        )

        #================================================
        # R² AJUSTADO
        #================================================

        n = len(y_real)

        r2_adj = 1 - (

            (1 - r2)

            * (n - 1)

            / (n - p - 1)
        )

        #================================================
        # RESULTADOS
        #================================================

        resultados.append({

            "Modelo":

                nombres[i],

            "R_square":

                round(r2, redondeo),

            "R_square_ajustado":

                round(r2_adj, redondeo),

            "MSE":

                round(mse, redondeo),

            "RMSE":

                round(rmse, redondeo),

            "MAE":

                round(mae, redondeo)
        })

    #----------------------------------------------------
    # DataFrame final
    #----------------------------------------------------

    resultados = pd.DataFrame(

        resultados
    )

    #----------------------------------------------------
    # Mostrar
    #----------------------------------------------------

    print("\n============================")

    print("Evaluación de Modelos")

    print("============================")

    print(resultados)

    return resultados

## Cargar datos

Se cargan los datos llamando la función *f_cargar_atos()* y la *url* en donde precisamente está el conjunto de datos de rendimiento académico. https://raw.githubusercontent.com/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/refs/heads/main/datos/datos%20educacion%20para%20regresiones%20multiple%20polinomiales%20rendimiento_academico.csv .





In [74]:
url = "https://raw.githubusercontent.com/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/refs/heads/main/datos/datos%20educacion%20para%20regresiones%20multiple%20polinomiales%20rendimiento_academico.csv"
datos = f_cargar_datos(url)


## Visualización de datos

In [75]:
f_visualizar_head_tail_reducido(datos)

,horas_estudio,tiempo_lectura,horas_sueno,uso_redes,...,nivel_socioeconomico,apoyo_familiar,estres,rendimiento_academico
0,9,9,8,6,...,3,8,5,77.96
1,1,1,6,4,...,5,4,1,81.15
2,6,6,9,7,...,2,1,4,84.74
3,4,4,5,6,...,5,8,5,84.77
4,5,5,6,5,...,3,2,2,82.41
5,10,10,5,8,...,5,4,5,79.28
6,...,...,...,...,...,...,...,...,...
7,4,3,5,5,...,4,5,4,82.79
8,4,5,9,3,...,1,7,10,85.53
9,7,6,5,1,...,4,1,4,87.01


## Estadísticos descriptivos

In [76]:
f_describir_datos(datos)

{'describe':        horas_estudio  tiempo_lectura  horas_sueno    uso_redes   motivacion  \
 count    6000.000000     6000.000000  6000.000000  6000.000000  6000.000000   
 mean        5.503667        5.520667     6.485667     4.540500     5.455500   
 std         2.908332        2.915703     1.701453     2.296281     2.887457   
 min         1.000000        1.000000     4.000000     1.000000     1.000000   
 25%         3.000000        3.000000     5.000000     3.000000     3.000000   
 50%         5.000000        6.000000     7.000000     5.000000     5.000000   
 75%         8.000000        8.000000     8.000000     7.000000     8.000000   
 max        10.000000       10.000000     9.000000     8.000000    10.000000   
 
         asistencia  nivel_socioeconomico  apoyo_familiar       estres  \
 count  6000.000000           6000.000000     6000.000000  6000.000000   
 mean     80.018333              2.990667        5.603333     5.496000   
 std      11.786736              1.427207   

# Desarrollo



## Partición de datos

Se generan los dos conjuntos de datos, *70%* para datos de entrenamiento y *30%* para datos de validación los cuales se usarán para crear y evaluar modelos de regesión lineal múltiple y polinomial múltiples de grado dos y tres.


In [77]:
particion = f_particionar_datos(datos)

datos_entrenamiento = particion["datos_entrenamiento"]
datos_validacion = particion["datos_validacion"]

f_visualizar_head_tail_reducido(datos_entrenamiento)

,horas_estudio,tiempo_lectura,horas_sueno,uso_redes,...,nivel_socioeconomico,apoyo_familiar,estres,rendimiento_academico
0,6,5,8,1,...,5,7,6,79.38
1,7,9,5,7,...,1,5,3,78.11
2,4,4,7,4,...,2,4,10,81.49
3,7,9,9,5,...,3,7,2,86.71
4,5,5,7,2,...,5,2,5,82.72
5,5,5,5,6,...,1,7,2,79.95
6,...,...,...,...,...,...,...,...,...
7,1,1,8,5,...,3,3,10,77.26
8,7,7,7,7,...,3,4,9,81.81
9,2,5,5,4,...,1,10,4,83.48


Datos de validación:



In [78]:
f_visualizar_head_tail_reducido(datos_validacion)

,horas_estudio,tiempo_lectura,horas_sueno,uso_redes,...,nivel_socioeconomico,apoyo_familiar,estres,rendimiento_academico
0,6,5,5,3,...,4,1,9,81.82
1,9,8,5,6,...,2,8,8,93.28
2,4,2,8,2,...,5,1,1,79.05
3,1,1,4,3,...,4,8,2,77.69
4,8,8,6,4,...,5,7,5,76.26
5,5,6,8,5,...,3,10,1,84.59
6,...,...,...,...,...,...,...,...,...
7,4,4,6,2,...,4,7,2,81.73
8,1,1,8,3,...,4,9,7,81.64
9,9,9,6,3,...,2,5,9,88.67


### Crear modelo de regresión lineal múltiple

Se manda llamar la función *f_construir_modelo_RLM_statsmodels()* con los datos de entrenamiento y la variable dependiente con la cual se construye el modelo de regresión lineal mútiple.


In [79]:
modelo_RLM = f_construir_modelo_RLM_statsmodels(datos_entrenamiento, "rendimiento_academico")


                              OLS Regression Results                             
Dep. Variable:     rendimiento_academico   R-squared:                       0.887
Model:                               OLS   Adj. R-squared:                  0.886
Method:                    Least Squares   F-statistic:                     3639.
Date:                   Wed, 20 May 2026   Prob (F-statistic):               0.00
Time:                           21:14:50   Log-Likelihood:                -8343.6
No. Observations:                   4200   AIC:                         1.671e+04
Df Residuals:                       4190   BIC:                         1.677e+04
Df Model:                              9                                         
Covariance Type:               nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
co

### Postulados del modelo de regesión lineal múltiple

El conjunto de datos y específicamente las variables independientes, presentan de manera moderada multicolinealidad entre ellas, además, el modleo no cumple con los postulados de linealidad ni normalidad en los resiudos.


In [80]:
f_validar_postulados_modelos (modelo_RLM)


Validación de Postulados
     Modelo Multicolinealidad  VIF_Max Linealidad Homocedasticidad Normalidad  \
0  Modelo_1          Moderada   9.1437  No cumple           Cumple  No cumple   

  Independencia  
0        Cumple  


,Modelo,Multicolinealidad,VIF_Max,Linealidad,Homocedasticidad,Normalidad,Independencia
0,Modelo_1,Moderada,9.1437,No cumple,Cumple,No cumple,Cumple


### Crear modelo de regresión polinomial de segundo orden



In [81]:
modelo_RPM_2 = f_multiple_polinomial(datos_entrenamiento, "rendimiento_academico", 2)


MODELO POLINOMIAL GRADO 2
Número de variables originales: 9
Número de términos polinomiales: 54
                              OLS Regression Results                             
Dep. Variable:     rendimiento_academico   R-squared:                       0.996
Model:                               OLS   Adj. R-squared:                  0.995
Method:                    Least Squares   F-statistic:                 1.700e+04
Date:                   Wed, 20 May 2026   Prob (F-statistic):               0.00
Time:                           21:14:50   Log-Likelihood:                -1564.2
No. Observations:                   4200   AIC:                             3238.
Df Residuals:                       4145   BIC:                             3587.
Df Model:                             54                                         
Covariance Type:               nonrobust                                         
                                          coef    std err          t      P>|t|    

### Crear modelo de regresión polinomial de tercer orden

Dada la complejidad de la cosntruccón del modelos polinomial múltiple de tercer orden el tiempo es un factor importante a considerar, en el equipo en donde fue ejecutado la función



In [82]:
modelo_RPM_3 = f_multiple_polinomial(datos_entrenamiento, "rendimiento_academico", 3)



MODELO POLINOMIAL GRADO 3
Número de variables originales: 9
Número de términos polinomiales: 219
                              OLS Regression Results                             
Dep. Variable:     rendimiento_academico   R-squared:                       0.996
Model:                               OLS   Adj. R-squared:                  0.996
Method:                    Least Squares   F-statistic:                     4490.
Date:                   Wed, 20 May 2026   Prob (F-statistic):               0.00
Time:                           21:14:50   Log-Likelihood:                -1335.5
No. Observations:                   4200   AIC:                             3111.
Df Residuals:                       3980   BIC:                             4506.
Df Model:                            219                                         
Covariance Type:               nonrobust                                         
                                                         coef    std err          

### Validar modelos

Se validan los tres modelos ejecutando la función f_validar_postulados_modelos() con los argumentos adecuados: la lista de los modelos, las variables independientes y la variable dependiente así como los nombres de cada modelo, la salida en modo consola.

  
En los resultados, se identifica que ningún modelo resuelve el problema de multicolinealidad, sin embargo, los modelos polinomiales garantizan validez en los postulados de linealidad, homocedasticidad, normalidad e independencia de residuos en comparación con el modelo de regresión lineal múltiple.

Aquí es importante hacer notar que el tiempo para la validación de todos lo postulados resultó algo considerable aproximadamente 98 segundos tal vez minuto y medio en el equipo personal del autor, una laptop con procesador 12th Gen Intel(R) Core(TM) i9-12950HX (2.30 GHz), 32.0 GB (31.7 GB usable) y tarjeta gráfica NVIDIA RTX A3000 12GB Laptop GPU (12 GB) Intel(R) UHD Graphics (128 MB).

Con lo anterior, será una decisión del lector, validar postulado por postulado como se realizó y describió con los anteriores casos de estudio, finalmente ya se tienen las funciones para validar de manera independiente estos postulados y que se encuentran en la dirección url al principio del caso citada.


In [ ]:
import time
# INICIO
inicio = time.time()
X = datos_entrenamiento.drop(columns = ["rendimiento_academico"]) # Variables independientes

y = datos_entrenamiento["rendimiento_academico"] # Variable dependiente

f_validar_postulados_modelos ([modelo_RLM, modelo_RPM_2, modelo_RPM_3],
                              X,
                              y,
                              nombres = ["RLM Original","Polinomial Múltiple 2","Polinomial Múltiple 3"])
fin = time.time()
tiempo = fin - inicio

print(

    f"\nTiempo de validación de postulados: "
    f"{round(tiempo,4)} segundos"
)

### Evaluar modelos

Se ejecuta la función *f_evaluacion_modelos()* la cual devuelve la evaluación de los estadísticos que permiten valorar la calidad predictiva de los modelos.

Se observa que los modelos polinomiales ofrecen mejor calidad predictiva en comparación con el modelo de regresión lineal múltiple con este conjunto de datos de validación.



In [ ]:
# INICIO
inicio = time.time()
f_evaluacion_modelos([modelo_RLM, modelo_RPM_2, modelo_RPM_2],
    datos_validacion,
    "rendimiento_academico",
    nombres = ["Lineal Múltiple", "Poli Múltiple 2", "Poli Múltiple 2"]
)
fin = time.time()
tiempo = fin - inicio

print(

    f"\nTiempo de ejecución: "
    f"{round(tiempo,4)} segundos"
)

## Construir modelos con datos normalizados

Para disminuir y regular la deficiencia de la multicolinealidad se construyen modelos con datos estandarizados, primero modelos de regresión lineal múltiple, luego, los modelos Lasso y Ridge, estos últimos con la finalidad de ser la alternativa para penalizar y regular la deficiencia de la muticolinealidad en las variables independientes.

Estos modelos se construyen con datos normalizados.

### Normalizar datos

Ahora se manda llamar la función f_ para normalziar los datos originales con los cuales se crean los nuevos datos de entrenamiento y validación para contruir los modelos regresión múltiple, *Lasso* y *Ridge*.





In [ ]:
datos_estandarizados_escalados = f_estandarizar_escalar(datos)
datos_estandarizados = datos_estandarizados_escalados['datos_estandarizados']
# datos_escalados = datos_estandarizados_escalados['datos_escalados']

particion = f_particionar_datos(datos_estandarizados) # Particionar con datos estandarizados

datos_entrenamiento = particion["datos_entrenamiento"]
datos_validacion = particion["datos_validacion"]

f_visualizar_head_tail_reducido(datos_entrenamiento)


In [ ]:
f_visualizar_head_tail_reducido(datos_validacion)

### Modelo de regresión lineal múltiple con datos normalizados



In [ ]:
modelo_RLM_norm = f_construir_modelo_RLM_statsmodels(datos_entrenamiento, "rendimiento_academico")

### Modelo Lasso



In [ ]:
modelo_lasso = f_construir_modelo_lasso(datos_entrenamiento, "rendimiento_academico")

### Modelo Ridge



In [ ]:
modelo_ridge = f_construir_modelo_ridge(datos_entrenamiento, "rendimiento_academico")

### Validación de modelos



In [ ]:
X = datos_entrenamiento.drop(columns = ["rendimiento_academico"]) # Variables independientes

y = datos_entrenamiento["rendimiento_academico"] # Variable dependiente

f_validar_postulados_modelos ([modelo_RLM_norm, modelo_lasso, modelo_ridge],
                              X, y,
                              nombres = ["RLM Normalizado","Lasso","Ridge"])

### Evaluación de modelos


In [ ]:
f_evaluacion_modelos([modelo_RLM_norm, modelo_lasso, modelo_ridge],
    datos_validacion,
    "rendimiento_academico",
    nombres = ["RLM", "LASSO", "RIDGE"]
)

# Interpretación del caso de estudio

El caso tuvo la finalidad de crear, evaluar y comparar modelos de regresión lineal múltiple con modelos de regresión polinomial múltiple de segundo y tercer grado. Se cumple con el objetivo del caso inicialmente definido.

Con datos originales se crearon particiones del 70% para datos de entrenamiento y 30% para datos de validación; se crearon tres modelos: de regresión lineal múltiple polinomiales múltiples de segundo y tercer orden.

El modelo de regresión múltiple con datos originales presenta de manera moderada el postulado de multicolinealidad y no favores al postulado de linealidad y normalidad; sin embargo, si ofrece calidad predictiva alrededor del 88% en el estadístico r square ajustado. Los modelos polinomiales también adolecen del postulado de multicolinealidad, pero garantizan validez en los demás postulados y ofrecen mejor calidad predictiva con alrededor del 99% en el estadístico; sin embargo, si ofrece calidad predictiva alrededor del 88% en el estadístico r square ajustado.

Luego, se transformaron los datos originales a datos estandarizados; con estos datos estandarizados de rendimiento académico, se crearon modelo de regresión lineal múltiple y modelos regularizados Lasso y Ridge.

Los resultados son similares en los estadísticos de evaluación sin embargo la idea de los modelos regularizados es que ayudan a una mejor estabilidad en los coeficientes penalizando los coeficientes dada la multicolinealidad de las variables.

Los resultados son similares a los que se tienen en el caso de estudio con programación *R* y finalmente cumplen con la métrica inicialmente establecida de que los modelos son satisfactorios si tiene un valor de r square ajustado por encima del *85%*.
